In [1]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

from scipy  import integrate
import astropy.units as u
import glob
from scipy.optimize import minimize
from gammapy.stats.fit_statistics import wstat, cash

plt.rc('xtick', labelsize=20)   
plt.rc('ytick', labelsize=20)
plt.rc('text', usetex=True)
plt.rc('font', family='serif',size=25)

from gammapy.data import Observation, Observations
from gammapy.modeling.models import (
    EBLAbsorptionNormSpectralModel,
    PowerLawSpectralModel,
    SpectralModel,
    ExpCutoffPowerLawSpectralModel,
    LogParabolaSpectralModel,
    TemplateSpectralModel
)

import os
print(os.getenv("GAMMAPY_DATA"))
os.environ["GAMMAPY_DATA"] = "/home/ddore/Documents/Omega-Centauri/damspi-main/gammapy-datasets/1.0/"


None


In [3]:
## Define your binninng in reconstructed energy EJ′ , in reconstructed time tI′ , and true time tI (the latter can coincide with the reco time)
all_obs = []
for run in sorted(glob.glob("./DL3_files/20*fits")): 
    print(run)
    obs = Observation.read(run) 
    all_obs.append(obs)

# migra = E_reco / E_true
all_migra       = np.array([obs.edisp.axes['migra'].bounds.value for obs in all_obs])
migra_min       = np.min(all_migra[:, 0])
migra_max       = np.max(all_migra[:, 1])
E_reco = np.geomspace(10*u.GeV, 10*u.TeV, 11)
E_true = np.geomspace(E_reco[0] / migra_max, E_reco[-1] / migra_min,200 )

# not linspace because not regular ? 
t_reco_left  = np.array([0, 118.25297, 236.50594, 354.7589, 473.01187, 591.26484, 709.51781, 827.77078, 946.02374, 1064.2767, 
          1182.5297, 1300.7826, 1419.0356, 1537.2886, 1655.5416, 1773.7945, 1892.0475, 2010.3005, 2128.5534, 
          2246.8064, 2365.0594, 2483.3123, 2601.5653, 2719.8183, 2838.0712, 2956.3242, 3074.5772, 3192.8301, 
          3311.0831, 3429.3361, 3547.589, 3665.842, 3784.095, 3902.3479, 4020.6009, 4138.8539, 4257.1068, 
          4375.3598, 4493.6128, 4611.8657, 4730.1187, 4848.3717, 4966.6247, 5084.8776, 5203.1306, 5321.3836, 
          5439.6365, 5557.8895, 5676.1425, 5794.3954, 5912.6484, 6030.9014, 6149.1543, 6267.4073, 6385.6603, 
          6503.9132, 6622.1662, 6740.4192, 6858.6721, 6976.9251, 7095.1781, 7213.431, 7331.684, 7449.937, 7568.1899, 
          7686.4429, 7804.6959, 7922.9489, 8041.2018])*u.s

t_reco_right = np.array( [ 118.25297, 236.50594, 354.7589, 473.01187, 591.26484, 709.51781, 827.77078, 946.02374, 1064.2767,
                       1182.5297, 1300.7826, 1419.0356, 1537.2886, 1655.5416, 1773.7945, 1892.0475, 2010.3005, 2128.5534,
                       2246.8064, 2365.0594, 2483.3123, 2601.5653, 2719.8183, 2838.0712, 2956.3242, 3074.5772, 3192.8301,
                       3311.0831, 3429.3361, 3547.589, 3665.842, 3784.095, 3902.3479, 4020.6009, 4138.8539, 4257.1068, 
                       4375.3598, 4493.6128, 4611.8657, 4730.1187, 4848.3717, 4966.6247, 5084.8776, 5203.1306, 5321.3836,
                       5439.6365, 5557.8895, 5676.1425, 5794.3954, 5912.6484, 6030.9014, 6149.1543, 6267.4073, 6385.6603,
                       6503.9132, 6622.1662, 6740.4192, 6858.6721, 6976.9251, 7095.1781, 7213.431, 7331.684, 7449.937, 
                       7568.1899, 7686.4429, 7804.6959, 7922.9489, 8041.2018, 8159.4548])*u.s

t_reco = np.append( t_reco_left, t_reco_right[-1] ) 

# Binning in observed and intrinsic time is the same (but can be different in principle)
t = t_reco 

print(migra_max, migra_min)
        

'THETA' axis is stored as a scalar -- converting to 1D array.
'THETA' axis is stored as a scalar -- converting to 1D array.


./DL3_files/20140426_05034767_DL3_Mrk421-W0.40+090.fits


'THETA' axis is stored as a scalar -- converting to 1D array.
'THETA' axis is stored as a scalar -- converting to 1D array.
'THETA' axis is stored as a scalar -- converting to 1D array.
'THETA' axis is stored as a scalar -- converting to 1D array.
'THETA' axis is stored as a scalar -- converting to 1D array.
'THETA' axis is stored as a scalar -- converting to 1D array.
'THETA' axis is stored as a scalar -- converting to 1D array.


./DL3_files/20140426_05034768_DL3_Mrk421-W0.40+270.fits
./DL3_files/20140426_05034771_DL3_Mrk421-W0.40+270.fits
./DL3_files/20140426_05034772_DL3_Mrk421-W0.40+000.fits


'THETA' axis is stored as a scalar -- converting to 1D array.
'THETA' axis is stored as a scalar -- converting to 1D array.
'THETA' axis is stored as a scalar -- converting to 1D array.
'THETA' axis is stored as a scalar -- converting to 1D array.
'THETA' axis is stored as a scalar -- converting to 1D array.
'THETA' axis is stored as a scalar -- converting to 1D array.
'THETA' axis is stored as a scalar -- converting to 1D array.
'THETA' axis is stored as a scalar -- converting to 1D array.
'THETA' axis is stored as a scalar -- converting to 1D array.


./DL3_files/20140426_05034773_DL3_Mrk421-W0.40+180.fits
./DL3_files/20140426_05034774_DL3_Mrk421-W0.40+090.fits
./DL3_files/20140426_05034775_DL3_Mrk421-W0.40+270.fits


'THETA' axis is stored as a scalar -- converting to 1D array.
'THETA' axis is stored as a scalar -- converting to 1D array.
'THETA' axis is stored as a scalar -- converting to 1D array.
'THETA' axis is stored as a scalar -- converting to 1D array.
'THETA' axis is stored as a scalar -- converting to 1D array.
'THETA' axis is stored as a scalar -- converting to 1D array.
'THETA' axis is stored as a scalar -- converting to 1D array.
'THETA' axis is stored as a scalar -- converting to 1D array.
'THETA' axis is stored as a scalar -- converting to 1D array.


./DL3_files/20140426_05034776_DL3_Mrk421-W0.40+000.fits
./DL3_files/20140426_05034777_DL3_Mrk421-W0.40+180.fits
10.0 0.10000000149011612


In [4]:
##Define your geometry for numerical integration for reconstructed energy Ej′ and true energy Ej

E_reco_num =  np.geomspace(E_reco[0], E_reco[-1],(len(E_reco)-1) * 20 + 1)
E_true_num =  np.geomspace(E_reco_num[0] / migra_max, E_reco_num[-1] / migra_min,200)


In [5]:
# this function is used for performing the numerical integration over  E' of G(E,E')
def integrate_over_each_range(integrand, bins, Delta):
    """
    Integrate a given integrand over specified ranges.

    This function performs numerical integration of a provided integrand array over
    each sub-interval defined by `Delta` within the `bins`. Integration is carried
    out using Simpson's rule.

    Parameters:
    integrand (np.ndarray): The values to be integrated. Assumed to be a 3D array,
                            with the integration performed along the second axis.
    bins (np.ndarray): The x-values corresponding to the integrand. These are the
                       points at which the integrand has been evaluated.
    Delta (np.ndarray): The edges of the sub-intervals over which to integrate.
                        Must be sorted in ascending order.

    Returns:
    np.ndarray: An array of integrated values for each sub-interval defined in `Delta`.

    Example:
    >>> integrand = np.random.rand(10, 100, 5)  # For example purposes
    >>> bins = np.linspace(0, 10, 100)  # x-values where integrand is evaluated
    >>> Delta = np.array([1, 3, 5, 7])  # Edges of sub-intervals
    >>> integrate_over_each_range(integrand, bins, Delta)
    array([...])  # Returns an array of integrated values
    """
    
    # axis = 1, when integrating E_prime in the first step
    axis = 1
    # Find the indices in bins that match the edges in Delta
    indices = np.searchsorted(bins, Delta)
    
    # Initialize an array to store the results of the integration over each range
    integrals = []

    # Iterate over the ranges defined by Delta
    for i in range(len(Delta)-1):
        # Define the start and end index for the current range
        start_idx = indices[i]
        end_idx = indices[i+1]
        
        # Extract the integrand values and corresponding bins for the current range
        y_values = integrand[:, start_idx:end_idx, :]
        x_values = bins[start_idx:end_idx]
        
        # Compute the integral for the current range using Simpson's rule
        integral = integrate.simpson(y_values, x=x_values, axis=axis)
        integrals.append(integral)

    return np.array(integrals)


In [ ]:
#Get from gammpay the IRF for each time bin I′ and integrate over reco. energy E′

#grille de valeur de milieu de bin logarithmique 
E_true_num_mid    = np.sqrt(E_true_num[:-1] * E_true_num[1:]) 
E_reco_num_mid = np.sqrt(E_reco_num[:-1] * E_reco_num[1:])

#limite 
E_true_min = E_reco_num_mid[:, None] / migra_max
E_true_max = E_reco_num_mid[:, None] / migra_min

#boolean converti en 0 ou 1
mask_migra = (E_true_num_mid[None, :] >= E_true_min) * (E_true_num_mid[None, :] <= E_true_max)

for obs in all_obs :
    edisp2D = obs.edisp #probability 
    edisp2D.normalize()
    E_reco = E_reco[:, None]

    edisp = edisp2D.evaluate(offset=0.4*u.deg, energy_true=E_true, migra=E_reco/E_true) / E_true #matrix
    edisp      = edisp * mask_migra 

    aeff = obs.aeff.evaluate( energy_true=E_true[None,:])
    aeff       = aeff * mask_migra

    IRF = aeff * edisp 
    # .T : fait une transposition, donc on passe à la forme (E_reco, E_true) (on inverse les axes) 
    IRF_grid = IRF.mean(axis=0).T[:,:,None] #self.IRF_grid has shape (E, E',t')

    edisp_matrix= integrate_over_each_range(integrand = IRF_grid, bins= E_reco.value, Delta = E_reco_num.value)

AttributeError: 'numpy.ndarray' object has no attribute 'to'

In [26]:
# this function is needed for performing the integeral over time and get the "time migration" matrix
def step_function_integral(a, b, a_prime, b_prime):
    """
    Calculate the integral of a step function over specified intervals.

    This function computes the integral of a step function that is 1 within the
    intervals defined by vectors a and b, and 0 otherwise, over another set of
    intervals defined by a_prime and b_prime. The function handles vectorized inputs
    and computes the overlap of each pair of intervals from the two sets.

    Parameters:
    a (np.ndarray): A 1D array of start points of the intervals where the step function is 1.
    b (np.ndarray): A 1D array of end points of the intervals where the step function is 1.
                    Must be the same shape as a.
    a_prime (np.ndarray): A 2D array of start points of the intervals over which to integrate.
    b_prime (np.ndarray): A 2D array of end points of the intervals over which to integrate.
                          Must be the same shape as a_prime.

    Returns:
    np.ndarray: A 3D array of the integral of the step function over each interval defined by
                a_prime and b_prime. The shape of the output array is based on the broadcasting
                rules applied to the input shapes.

    Example:
    >>> a = np.array([0, 5])
    >>> b = np.array([3, 10])
    >>> a_prime = np.array([[1, 6], [4, 8]])
    >>> b_prime = np.array([[2, 7], [5, 9]])
    >>> step_function_integral(a, b, a_prime, b_prime)
    array([[[ 1,  1],
            [ 0,  2]],

           [[ 2,  3],
            [ 1,  2]]])
    """
    
    # Ensure inputs are numpy arrays for vectorized operations
    a = np.asarray(a)
    b = np.asarray(b)
    a_prime = np.asarray(a_prime)
    b_prime = np.asarray(b_prime)
    
    # Calculate the overlap between [a, b] and [a_prime, b_prime] using broadcasting
    overlap_start = np.maximum(a[None,None,:], a_prime[:,:,None])
    overlap_end = np.minimum(b[None,None,:], b_prime[:,:,None])

    # The result is the positive length of the overlap, or 0 where there is no overlap
    overlap_length = np.maximum(0, overlap_end - overlap_start)

    return overlap_length


In [ ]:
#Compute the “time migration matrix”

time_matrix = step_function_integral(a=t_reco_i , b=t_reco_f, a_prime= t_i + eta* E_true, b_prime = t_f + eta*E_true)

# time matrix LIV + edisp matrix IRF 

migr_matrix = time_matrix*edisp_matrix

In [ ]:
#Assuming a given spectral shape in true energy Φ̃j and assuming amplitude FI for each time bin, compute the expected signal count 
z = 0.03
model_EBL = EBLAbsorptionNormSpectralModel.read_builtin("dominguez", redshift = z )
spectral_model_EBL = LogParabolaSpectralModel(amplitude = 7e-9 / (u.cm**2 *u.s * u.TeV), alpha=2, beta= 0.1, reference= 202*u.GeV)*model_EBL
#va chercher les meilleurs valeur de amp_F qui reproduisent les données pour un eta fixe. 
# puis apres en balayant sur les eta on regarde quelle valeur de eta a la meilleure minimisation = donne le retard
amp_F = np.ones_like(t_reco_left) 
limit = [(1e-100,None) for kk in range(len(am_F))]

sum_over_t = np.sum(spectral_model_EBL(E_true)*amp_F*migr_matrix)
expected_sig_count = integrate.simpson(sum_over_t_, E_true)
# calculation of log(likelihood)with wstat
n_on = np.load("./ON_counts.npy")
n_off = np.load("./OFF_counts.npy")
alpha = 1/3 #Facteur d’exposition entre zone ON et OFF
log_lkl = np.sum(wstat(n_on = n_on, n_off=n_off, alpha= alpha, mu_sig=expected_sig_count))


In [ ]:
# minimize log lkl 
solution = minimize(log_lkl,x0 = amp_F, bounds= limit, method= 'L-BFGS-B', options={'maxfun': 1000000,  'gtol': 1e-8})

In [ ]:
#loglkl_min 
solution = loglkl_min 